# Discrete Diffusion on 32x32 Mazes

Synthetic binary mazes are represented by fixed 16-dimensional embeddings. A Gaussian DDPM corrupts the continuous tensor; a U-Net predicts the added noise, and the final result is greedily decoded into walls and passages.

Run cells manually. Training may take a while.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "pyproject.toml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "notebooks"))

import numpy as np
import plotly.graph_objects as go
import torch
import importlib
from IPython.display import clear_output, display
from torch.nn import functional as F
from torch.utils.data import DataLoader, TensorDataset
from plotly.subplots import make_subplots

from _shared.maze_data import generate_dataset, solvability_rate
import _shared.model as model_module

importlib.reload(model_module)
from _shared.model import (
    Denoiser,
    decode_probabilities,
    embed_binary,
    make_binary_embeddings,
)

SEED = 7
torch.manual_seed(SEED)
np.random.seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Project: {PROJECT_ROOT} | device: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)} | CUDA runtime: {torch.version.cuda}")
else:
    print(
        f"CUDA unavailable to PyTorch (torch.version.cuda={torch.version.cuda}). Run uv sync and select the project .venv kernel."
    )

Project: c:\Users\shich\Src\discrete_diffusion | device: cuda
GPU: NVIDIA GeForce RTX 3050 6GB Laptop GPU | CUDA runtime: 12.8


## 1. Synthetic Dataset

Randomized DFS generates perfect mazes. 0 means wall; 1 means passage.

In [8]:
N_MAZES = 12_000
VAL_FRACTION = 0.1
mazes = generate_dataset(N_MAZES, seed=SEED)
n_val = int(N_MAZES * VAL_FRACTION)
train_mazes = torch.from_numpy(mazes[:-n_val].copy()).long()
val_mazes = torch.from_numpy(mazes[-n_val:].copy()).long()
print(
    "all:", mazes.shape, "| train:", len(train_mazes), "| validation:", len(val_mazes)
)
print(f"Generated-maze solvability: {solvability_rate(mazes):.1%}")

fig = go.Figure(
    go.Heatmap(
        z=mazes[0],
        zmin=0,
        zmax=1,
        colorscale=[[0, "black"], [1, "white"]],
        showscale=False,
    )
)
fig.update_layout(
    title="Synthetic maze example",
    width=420,
    height=420,
    yaxis=dict(autorange="reversed", scaleanchor="x"),
)
fig.show()

# DDPM configuration. Pixel classes stay embedded in 16 dimensions.
DIFFUSION_STEPS = 100
EMBEDDING_DIM = 2
TEMPERATURE = 0.5


def cosine_beta_schedule(num_steps, offset=0.008, device=DEVICE):
    grid = torch.linspace(
        0, num_steps, num_steps + 1, device=device, dtype=torch.float32
    )
    cumulative = torch.cos(
        ((grid / num_steps + offset) / (1 + offset)) * torch.pi / 2
    ).square()
    cumulative = cumulative / cumulative[0]
    return (1 - cumulative[1:] / cumulative[:-1]).clamp(1e-5, 0.999)


betas = cosine_beta_schedule(DIFFUSION_STEPS)
alphas = 1.0 - betas
alpha_bars = torch.cumprod(alphas, dim=0)


def q_sample(x0, timesteps, noise=None):
    """Sample q(x_t|x_0) directly; no per-pixel normalization is applied."""
    if noise is None:
        noise = torch.randn_like(x0)
    a_bar = alpha_bars[timesteps].view(-1, 1, 1, 1)
    return a_bar.sqrt() * x0 + (1 - a_bar).sqrt() * noise, noise


binary_embeddings = make_binary_embeddings(EMBEDDING_DIM, DEVICE)
print("Fixed unit embeddings [wall=0, passage=1], dimension 16:")
print(binary_embeddings.detach().cpu())
print(
    f"Embedding cosine similarity: {torch.dot(binary_embeddings[0], binary_embeddings[1]).item():.2e}"
)

preview_maze = torch.from_numpy(mazes[0].copy()).long().unsqueeze(0).to(DEVICE)
preview_x0 = embed_binary(preview_maze, binary_embeddings)
preview_noise = torch.randn_like(preview_x0)
preview_timesteps = [0, 1, 2, 4, 7, 11, 16, 22, 30, 39, 49, 59, 69, 79, 89, 94, 97, 99]
with torch.no_grad():
    preview_frames = [
        decode_probabilities(preview_x0, binary_embeddings, TEMPERATURE)[0, 1]
        .cpu()
        .numpy()
    ]
    for step in preview_timesteps:
        timestep = torch.tensor([step], device=DEVICE)
        noisy, _ = q_sample(preview_x0, timestep, preview_noise)
        probabilities = decode_probabilities(noisy, binary_embeddings, TEMPERATURE)
        preview_frames.append(probabilities[0, 1].cpu().numpy())

progress_fig = make_subplots(
    rows=2,
    cols=10,
    subplot_titles=[f"t={t}" for t in [0] + [step + 1 for step in preview_timesteps]],
)
for frame_index, frame in enumerate(preview_frames):
    progress_fig.add_trace(
        go.Heatmap(
            z=frame,
            zmin=0,
            zmax=1,
            colorscale=[[0, "black"], [1, "white"]],
            showscale=False,
        ),
        row=frame_index // 10 + 1,
        col=frame_index % 10 + 1,
    )
progress_fig.update_layout(
    title="Forward DDPM: black = p(0), white = p(1)",
    width=1700,
    height=650,
)
progress_fig.update_xaxes(showticklabels=False)
progress_fig.update_yaxes(showticklabels=False, autorange="reversed")
progress_fig.show()

all: (12000, 32, 32) | train: 10800 | validation: 1200
Generated-maze solvability: 100.0%


Fixed unit embeddings [wall=0, passage=1], dimension 16:
tensor([[1., 0.],
        [0., 1.]])
Embedding cosine similarity: 0.00e+00


## 2. Model Architecture

A timestep-conditioned U-Net predicts Gaussian noise `epsilon` from a noisy 16-channel maze and timestep `t`. Embeddings are not normalized inside the diffusion process or network. See [docs/architecture.typ](../docs/architecture.typ) for all layers and dimensions.

In [10]:
# Parameters match docs/architecture.typ
BASE_CHANNELS = 64
TIME_DIM = 128

model = Denoiser(
    embedding_dim=EMBEDDING_DIM,
    base_channels=BASE_CHANNELS,
    time_dim=TIME_DIM,
).to(DEVICE)
print(model)
print(
    f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}"
)

Denoiser(
  (time_mlp): Sequential(
    (0): Linear(in_features=128, out_features=512, bias=True)
    (1): SiLU()
    (2): Linear(in_features=512, out_features=128, bias=True)
  )
  (input): Conv2d(2, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (enc32): ResBlock(
    (norm1): GroupNorm(8, 64, eps=1e-05, affine=True)
    (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (time_projection): Linear(in_features=128, out_features=64, bias=True)
    (norm2): GroupNorm(8, 64, eps=1e-05, affine=True)
    (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (skip): Identity()
  )
  (down16): Conv2d(64, 128, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
  (enc16): ResBlock(
    (norm1): GroupNorm(8, 128, eps=1e-05, affine=True)
    (conv1): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (time_projection): Linear(in_features=128, out_features=128, bias=True)
    (norm2): GroupNorm(8, 128, eps=1e-05, a

## 3. Training

For each maze, sample a random timestep `t` and form `x_t = sqrt(alpha_bar_t) * x_0 + sqrt(1 - alpha_bar_t) * epsilon`. Train the U-Net to predict the added noise `epsilon` with MSE. Validation uses the same objective. The live loss chart updates in place.

In [11]:
BATCH_SIZE = 32
EPOCHS = 30
LEARNING_RATE = 2e-4
PLOT_EVERY = 100

train_loader = DataLoader(
    TensorDataset(train_mazes), batch_size=BATCH_SIZE, shuffle=True, drop_last=True
)
val_loader = DataLoader(TensorDataset(val_mazes), batch_size=BATCH_SIZE, shuffle=False)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)


def batch_loss(maze_batch):
    x0 = embed_binary(maze_batch.to(DEVICE), binary_embeddings)
    timesteps = torch.randint(0, DIFFUSION_STEPS, (x0.shape[0],), device=DEVICE)
    x_t, noise = q_sample(x0, timesteps)
    predicted_noise = model(x_t, timesteps)
    return F.mse_loss(predicted_noise, noise)


@torch.no_grad()
def validation_loss():
    model.eval()
    values = [batch_loss(batch).item() for (batch,) in val_loader]
    model.train()
    return float(np.mean(values))


model.train()
training_completed = False
train_history, val_history = [], []
loss_fig = go.Figure()
loss_fig.add_scatter(name="train", mode="lines")
loss_fig.add_scatter(name="validation", mode="lines+markers")
loss_fig.update_layout(
    title="DDPM epsilon-prediction MSE",
    xaxis_title="Optimization step",
    yaxis_title="MSE of predicted noise",
    yaxis_type="log",
)
global_step = 0
for epoch in range(EPOCHS):
    for (batch,) in train_loader:
        optimizer.zero_grad(set_to_none=True)
        loss = batch_loss(batch)
        loss.backward()
        optimizer.step()
        train_history.append(loss.item())
        global_step += 1
        if global_step % PLOT_EVERY == 0:
            loss_fig.data[0].x = list(range(1, len(train_history) + 1))
            loss_fig.data[0].y = train_history
            loss_fig.data[1].x = [step for step, _ in val_history]
            loss_fig.data[1].y = [value for _, value in val_history]
            loss_fig.update_layout(
                title=f"DDPM training | epoch {epoch + 1}/{EPOCHS} | step {global_step}"
            )
            clear_output(wait=True)
            display(loss_fig)
    epoch_train = float(np.mean(train_history[-len(train_loader) :]))
    epoch_validation = validation_loss()
    val_history.append((global_step, epoch_validation))
    loss_fig.data[0].x = list(range(1, len(train_history) + 1))
    loss_fig.data[0].y = train_history
    loss_fig.data[1].x = [step for step, _ in val_history]
    loss_fig.data[1].y = [value for _, value in val_history]
    loss_fig.update_layout(
        title=f"DDPM training | epoch {epoch + 1}/{EPOCHS} | step {global_step}"
    )
    clear_output(wait=True)
    display(loss_fig)
    print(
        f"Epoch {epoch + 1:02d}/{EPOCHS} | train={epoch_train:.4f} | val={epoch_validation:.4f}"
    )

training_completed = True
print(f"Training complete | train={epoch_train:.4f} | val={epoch_validation:.4f}")

Epoch 30/30 | train=0.0245 | val=0.0242
Training complete | train=0.0245 | val=0.0242


## 4. Save the Model

In [13]:
ARTIFACTS = PROJECT_ROOT / "artifacts"
ARTIFACTS.mkdir(parents=True, exist_ok=True)
checkpoint_path = ARTIFACTS / "maze_ddpm_epsilon.pt"
torch.save(
    {
        "model_state_dict": model.state_dict(),
        "binary_embeddings": binary_embeddings.detach().cpu(),
        "config": {
            "diffusion_steps": DIFFUSION_STEPS,
            "embedding_dim": EMBEDDING_DIM,
            "base_channels": BASE_CHANNELS,
            "time_dim": TIME_DIM,
            "beta_schedule": "cosine",
            "prediction_target": "epsilon",
        },
    },
    checkpoint_path,
)
print(f"Saved: {checkpoint_path}")

Saved: c:\Users\shich\Src\discrete_diffusion\artifacts\maze_ddpm_epsilon.pt


## 5. Reverse DDPM Sampling and Solvability

Sampling starts from `x_T ~ N(0, I)` and applies all reverse DDPM steps using the predicted noise. Clean-image estimates are captured at selected timesteps for visualization; they are not fed back into the network. Only the final continuous result is greedily decoded by dot product against the two fixed 16-dimensional embeddings.

In [12]:
N_METRIC_SAMPLES = 128
N_TO_SHOW = 4
REVERSE_PREVIEW_TIMESTEPS = [89, 74, 59, 44, 29, 19, 9, 4, 0]


@torch.no_grad()
def sample_ddpm(model, n_samples, capture_timesteps=()):
    model.eval()
    x = torch.randn(n_samples, EMBEDDING_DIM, 32, 32, device=DEVICE)
    initial_noise = x.detach().clone()
    snapshots = {}
    capture_timesteps = set(capture_timesteps)

    for step in reversed(range(DIFFUSION_STEPS)):
        timesteps = torch.full((n_samples,), step, device=DEVICE, dtype=torch.long)
        predicted_noise = model(x, timesteps)
        alpha = alphas[step]
        alpha_bar = alpha_bars[step]
        beta = betas[step]
        x0_estimate = (x - (1 - alpha_bar).sqrt() * predicted_noise) / alpha_bar.sqrt()
        if step in capture_timesteps:
            snapshots[step] = x0_estimate.detach().clone()

        mean = (x - beta / (1 - alpha_bar).sqrt() * predicted_noise) / alpha.sqrt()
        if step > 0:
            alpha_bar_previous = alpha_bars[step - 1]
            posterior_variance = beta * (1 - alpha_bar_previous) / (1 - alpha_bar)
            x = mean + posterior_variance.sqrt() * torch.randn_like(x)
        else:
            x = mean

    return x, snapshots, initial_noise


samples_continuous, reverse_snapshots, initial_noise = sample_ddpm(
    model, N_METRIC_SAMPLES, REVERSE_PREVIEW_TIMESTEPS
)
print(
    "Initial x_T Gaussian stats: "
    f"mean={initial_noise.mean().item():.3f}, "
    f"std={initial_noise.std().item():.3f}"
)
hard_samples = (
    torch.einsum("bdhw,kd->bkhw", samples_continuous, binary_embeddings)
    .argmax(dim=1)
    .cpu()
    .numpy()
    .astype(np.uint8)
)

fig = make_subplots(
    rows=N_TO_SHOW,
    cols=1 + len(REVERSE_PREVIEW_TIMESTEPS),
    subplot_titles=[
        f"t={t}"
        for _ in range(N_TO_SHOW)
        for t in [100] + [step + 1 for step in REVERSE_PREVIEW_TIMESTEPS]
    ],
)
for i in range(N_TO_SHOW):
    initial_probabilities = decode_probabilities(
        initial_noise[i : i + 1], binary_embeddings, TEMPERATURE
    )[0, 1]
    fig.add_trace(
        go.Heatmap(
            z=initial_probabilities.cpu().numpy(),
            zmin=0,
            zmax=1,
            colorscale=[[0, "black"], [1, "white"]],
            showscale=False,
        ),
        row=i + 1,
        col=1,
    )
    for col, step in enumerate(REVERSE_PREVIEW_TIMESTEPS, start=2):
        probabilities = (
            decode_probabilities(
                reverse_snapshots[step], binary_embeddings, TEMPERATURE
            )[i, 1]
            .cpu()
            .numpy()
        )
        fig.add_trace(
            go.Heatmap(
                z=probabilities,
                zmin=0,
                zmax=1,
                colorscale=[[0, "black"], [1, "white"]],
                showscale=False,
            ),
            row=i + 1,
            col=col,
        )
fig.update_layout(
    title="Reverse DDPM: black = p(0), white = p(1)",
    height=850,
    width=1900,
)
fig.update_xaxes(showticklabels=False)
fig.update_yaxes(showticklabels=False, autorange="reversed")
fig.show()

solved = [solvability_rate([maze]) for maze in hard_samples]
print(f"Solvable samples: {sum(solved)}/{len(solved)} ({np.mean(solved):.1%})")
hard_fig = make_subplots(
    rows=1,
    cols=N_TO_SHOW,
    subplot_titles=[
        f"Sample {i + 1}: {'solvable' if solved[i] else 'unsolvable'}"
        for i in range(N_TO_SHOW)
    ],
)
for i in range(N_TO_SHOW):
    hard_fig.add_trace(
        go.Heatmap(
            z=hard_samples[i],
            zmin=0,
            zmax=1,
            colorscale=[[0, "black"], [1, "white"]],
            showscale=False,
        ),
        row=1,
        col=i + 1,
    )
hard_fig.update_layout(title="Final mazes after greedy decoding", height=390, width=900)
hard_fig.update_xaxes(showticklabels=False)
hard_fig.update_yaxes(showticklabels=False, autorange="reversed")
hard_fig.show()

Initial x_T Gaussian stats: mean=-0.000, std=0.999


Solvable samples: 41.0/128 (32.0%)
